In [ ]:
import pandas as pd

data=pd.read_csv("SMSSpamCollection",
                 sep="\t",
                 header=None,
                 names=['labels','message'])
data.head()

,labels,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
data.isnull().sum()


,0
labels,0
message,0


In [ ]:
data.shape

(5572, 2)

In [ ]:
data=data.drop_duplicates()
data.shape

(5169, 2)

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5169 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   labels   5169 non-null   object
 1   message  5169 non-null   object
dtypes: object(2)
memory usage: 121.1+ KB


In [ ]:
import re
def text_preprocesing(text):
  text=text.lower()
  text=re.sub(r"http\S+|www\S+", "", text)

  text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)

  text = re.sub(r"\s+", " ", text).strip()
  return text
data["clean_message"] = data["message"].apply(text_preprocesing)

print(data[["message", "clean_message"]].head())

                                             message  \
0  Go until jurong point, crazy.. Available only ...   
1                      Ok lar... Joking wif u oni...   
2  Free entry in 2 a wkly comp to win FA Cup fina...   
3  U dun say so early hor... U c already then say...   
4  Nah I don't think he goes to usf, he lives aro...   

                                       clean_message  
0  go until jurong point crazy available only in ...  
1                            ok lar joking wif u oni  
2  free entry in 2 a wkly comp to win fa cup fina...  
3        u dun say so early hor u c already then say  
4  nah i don t think he goes to usf he lives arou...  


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
cv=CountVectorizer(stop_words="english")
x=cv.fit_transform(data["clean_message"])
y=data['labels']
print(x.shape)

(5169, 8382)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])

Training records: 4135
Testing records: 1034


In [ ]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()

In [ ]:
model.fit(X_train, y_train)

print("Model training completed successfully!")

Model training completed successfully!


In [ ]:
y_pred = model.predict(X_test)

print(y_pred[:10])

['ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham']


In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)

print("Model Accuracy:", accuracy * 100, "%")

# Classification report
print(classification_report(y_test, y_pred))

# Confusion matrix
print(confusion_matrix(y_test, y_pred))

Model Accuracy: 97.58220502901354 %
              precision    recall  f1-score   support

         ham       0.99      0.98      0.99       903
        spam       0.87      0.95      0.91       131

    accuracy                           0.98      1034
   macro avg       0.93      0.97      0.95      1034
weighted avg       0.98      0.98      0.98      1034

[[884  19]
 [  6 125]]


In [ ]:
def predict_spam(message):

    cleaned = text_preprocesing(message)

    vector = cv.transform([cleaned])

    prediction = model.predict(vector)

    if prediction[0] == "spam":
        return "Spam Message"

    else:
        return "Not Spam (Ham)"



print(predict_spam("Congratulations! You won a free prize."))
print(predict_spam("Hey, are you coming to college today?"))

Spam Message
Not Spam (Ham)


In [ ]:
predict_spam('Congratulations! You won a free prize.')

'Spam Message'

In [ ]:
!pip install gradio joblib


In [ ]:
import joblib

joblib.dump(model, "spam_detection_model.pkl")

print("Model saved successfully!")

Model saved successfully!


In [ ]:
joblib.dump(model, "naive_bayes_model.pkl")
joblib.dump(cv, "count_vectorizer.pkl")

['count_vectorizer.pkl']

In [ ]:
import gradio as gr

def predict_spam(message):

    if not message or not message.strip():
        return "Please enter a message."

    # Step 1: Preprocessing
    cleaned_message = text_preprocesing(message)

    # Step 2: Count Vectorization
    message_vector = cv.transform([cleaned_message])

    # Step 3: Multinomial Naive Bayes prediction
    prediction = model.predict(message_vector)[0]

    # Step 4: Calculate confidence
    probabilities = model.predict_proba(message_vector)[0]
    confidence = max(probabilities) * 100

    if prediction == "spam":
        return f"🚨 SPAM MESSAGE\nConfidence: {confidence:.2f}%"
    else:
        return f"✅ NOT SPAM (HAM)\nConfidence: {confidence:.2f}%"

In [ ]:
app = gr.Interface(
    fn=predict_spam,
    inputs=gr.Textbox(
        lines=4,
        label="Enter Your Message",
        placeholder="Type your SMS message here..."
    ),
    outputs=gr.Textbox(label="Detection Result"),
    title="SMS Spam Detection System",
    description="Spam detection using Text Preprocessing, Count Vectorization and Multinomial Naive Bayes.",
    examples=[
        ["Congratulations! You won a free prize."],
        ["Hey, are you coming to college today?"],
        ["URGENT! Claim your free cash reward now."]
    ]
)

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e3b8f3022963014a31.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
